Instalações bibliotecas

In [ ]:
!pip install google-adk -q
!pip install litellm -q

Importação das bibliotecas

In [ ]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For multi-model support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Libraries imported.")

Libraries imported.


Importação das chaves de API

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv() # Colocando as chaves de API em um arquivo .env, para que elas não vazem

print("Google:", bool(os.getenv("GOOGLE_API_KEY")))
print("OpenAI:", bool(os.getenv("OPENAI_API_KEY")))
print("Anthropic:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("Groq:", bool(os.getenv("GROQ_API_KEY")))

Google: True
OpenAI: True
Anthropic: True
Groq: True


Definindo os modelos que serão utilizados

In [ ]:
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

MODEL_GEMINI_2_5_FLASH_LITE = "gemini/gemini-2.5-flash-lite"

MODEL_GROQ = "groq/llama-3.3-70b-versatile"

MODEL_GPT_4O = "openai/gpt-4.1"

MODEL_CLAUDE_SONNET = "anthropic/claude-sonnet-4-20250514"

print("\nEnvironment configured.")


Environment configured.


Definindo a função 'get_weather'

In [ ]:
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_weather called for city: {city} ---") # Imprime qual cidade foi recebida
    city_normalized = city.lower().replace(" ", "") # Normaliza o nome da cidade, deixando ela em minúscula e sem espaços

    # Mock weather data
    mock_weather_db = { # dicionário que funciona como um banco de dados para o modelo
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    if city_normalized in mock_weather_db: # Verifica se a cidade recebida está no dicionário
        return mock_weather_db[city_normalized] # Se tiver retorna os respectivos dados daquela cidade
    else: # Se não tiver retorna uma mensagem de erro
        return {"status": "error", "error_message": f"Sorry, I don't have weather information for '{city}'."}

# Exemplo de teste
print(get_weather("New York")) # Tem a cidade no dicionário
print(get_weather("Paris")) # Não tem a cidade no dicionário

--- Tool: get_weather called for city: New York ---
{'status': 'success', 'report': 'The weather in New York is sunny with a temperature of 25°C.'}
--- Tool: get_weather called for city: Paris ---
{'status': 'error', 'error_message': "Sorry, I don't have weather information for 'Paris'."}


Definindo o Agente (weather_agent)

In [ ]:
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH # Começando com o modelo do Gemini para o weather_agent

weather_agent = Agent(
    name="weather_agent_v1", # Identificador único para o Agente
    model=AGENT_MODEL, # Especifica qual LLM utilizar
    description="Provides weather information for specific cities.", # Resumo conciso do propósito do Agente
    instruction="You are a helpful weather assistant. " # Orientações para o Agente
                "When the user asks for the weather in a specific city, "
                "use the 'get_weather' tool to find the information. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the weather report clearly.",
    tools=[get_weather], # Uma lista das ferramentas que estão disponíveis para o Agente utilizar
)

print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'weather_agent_v1' created using model 'gemini-2.5-flash'.


Configurando o Serviço de Sessão e o Runner

In [ ]:
# --- Gerenciamento de Sessão. ---
# Conceito principal: SessionService guarda o histórico e o estado da conversa.
# InMemorySessionService simples, não salva em banco.
session_service = InMemorySessionService()

APP_NAME = "weather_tutorial_app" # Nome da aplicação
USER_ID = "user_1" # Id do usuário, para distinguir sessões de usuários diferentes
SESSION_ID = "session_001" # Identificador fixo de sessão

session = await session_service.create_session( # Cria assíncronamente uma nova sessão
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Conceito principal: o Runner controla o ciclo de execução do agente.
runner = Runner(
    agent=weather_agent, # O agente que irá ser rodado
    app_name=APP_NAME,   # Associa as execuções do Runner à aplicação
    session_service=session_service # Usa o gerenciamento de sessão
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='weather_tutorial_app', User='user_1', Session='session_001'
Runner created for agent 'weather_agent_v1'.


Definindo a função de Interação do Agente

In [ ]:
from google.genai import types

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")


  content = types.Content(role='user', parts=[types.Part(text=query)]) # Prepara a mensagem do usuário no formado que o ADK exige

  final_response_text = "Agent did not produce a final response." # Default

  # run_async executa a lógica do agente e produz eventos
  # Percorre os eventos até encontrar a resposta final
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content): # Loop que executa o agente e recebe eventos gerados durante a execução
      print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}") # Permite ver todos os eventos durante a execução

      # is_final_response() indica o evento que encerra a resposta do agente
      if event.is_final_response(): # Verifica se o evento atual é a resposta final do agente
          if event.content and event.content.parts: # Garante que o evento contém conteúdo e que esse conteúdo possui partes
             # Assumindo que o texto da reposta está na primeira parte, extrai ele e armazena como reposta final
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Caso não haja texto, verifica erros
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          break # Interrompe o loop assim que a resposta final é encontrada

  print(f"<<< Agent Response: {final_response_text}")

Rodando a conversação incial

In [ ]:
# É necessário uma função assíncrona para usar await
async def run_conversation(): # Executa várias interações com o agente
    await call_agent_async("What is the weather like in London?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

    await call_agent_async("How about Paris?",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID) # Espera-se um erro da ferramenta

    await call_agent_async("Tell me the weather in New York",
                                       runner=runner,
                                       user_id=USER_ID,
                                       session_id=SESSION_ID)

# Executa a conversa usando await em um contexto assíncrono
await run_conversation()


>>> User Query: What is the weather like in London?
  [Event] Author: weather_agent_v1, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'city': 'London'
    },
    id='adk-2f5d8320-5ab3-4917-8aa2-f8ec424672df',
    name='get_weather'
  ),
  thought_signature=b"\n\x8f\x02\x01\xbe>\xf6\xfb\x1f\x8c3\xac<\xac\xcd\xaf^\xf5n\xb2\xe0I\xca\xdec\xf1{\x02s#\x86'\x91\x17-\x18\xc1\x15\xd8;p\x80-<\x9eD\xb0\xf2\x8b\xf8\x07\x11\xc7lX*\xf6E\xf8\xd6|\x13n\x92\xb8\x99\xe0]g*l\xbd6\x19\xe8{\x82\xaa\xd3\xf8\xa4\xbbp*4\x08m\xe6\xb6\xf9\xf0\xcf\xb32e+\xec...'
)] role='model'
--- Tool: get_weather called for city: London ---
  [Event] Author: weather_agent_v1, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='adk-2f5d8320-5ab3-4917-8aa2-f8ec424672df',
    name='get_weather',
    response={
      'report': "It's cloudy in London with a temperature of 15°C.",
      'status': 'success'
    }
  )
)] role='user'
  [Event

Importando o LiteLlm

In [ ]:
from google.adk.models.lite_llm import LiteLlm

Definindo e testando o Agente Gemini, nessa etapa era para se utilizar o agente GPT, mas ele não oferece nenhum modelo que possa ser utilizado gratuitamente

In [ ]:
# --- Agente usando o gemini-2.5-flash-lite ---
weather_agent_gemini = None # Inicializa a variável do agente como None
runner_gemini = None      # Inicializa a variável do runner como None

try:
    weather_agent_gemini = Agent(
        name="weather_agent_gemini", # Define um nome para o Agente
        model=LiteLlm(model=MODEL_GEMINI_2_5_FLASH_LITE), # Configura o agente para usar um modelo via LiteLLM
        description="Provides weather information (using gemini-2.5-flash-lite).", # Descrição curta do que o agente faz
        instruction="You are a helpful weather assistant powered by gemini-2.5-flash-lite. " # Instrução para orientar o agente
                    "Use the 'get_weather' tool for city weather requests. "
                    "Clearly present successful reports or polite error messages based on the tool's output status.",
        tools=[get_weather], # Usa novamente a mesma ferramenta
    )
    print(f"Agent '{weather_agent_gemini.name}' created using model '{MODEL_GEMINI_2_5_FLASH_LITE}'.")

    # InMemorySessionService simples, não salva em banco.
    session_service_gemini = InMemorySessionService() # Cria um servidor dedicado

    APP_NAME_GEMINI = "weather_tutorial_app_gemini" # Nome da aplicação
    USER_ID_GEMINI = "user_1_gemini" # Id do usuário, para distinguir sessões de usuários diferentes
    SESSION_ID_GEMINI = "session_001_gemini" # Identificador fixo de sessão

    # Cria a sessão específica a conversa irá ocorrer
    session_gemini = await session_service_gemini.create_session(
        app_name=APP_NAME_GEMINI,
        user_id=USER_ID_GEMINI,
        session_id=SESSION_ID_GEMINI
    )
    print(f"Session created: App='{APP_NAME_GEMINI}', User='{USER_ID_GEMINI}', Session='{SESSION_ID_GEMINI}'")

    # Agora cria o Runner específico desse agente e desse serviço de sessão
    runner_gemini = Runner( # Instancia o Runner que vai executar o agente em loop de eventos.
        agent=weather_agent_gemini,
        app_name=APP_NAME_GEMINI,       # Usa o nome da aplicação
        session_service=session_service_gemini # Usa a sessão específica
        )
    print(f"Runner created for agent '{runner_gemini.agent.name}'.")

    # --- Testa o gemini Agente ---
    print("\n--- Testing gemini Agent ---")
    await call_agent_async(query = "What's the weather in Tokyo?",
                           runner=runner_gemini,
                           user_id=USER_ID_GEMINI,
                           session_id=SESSION_ID_GEMINI)

except Exception as e:
    print(f"❌ Could not create or run gemini agent '{MODEL_GEMINI_2_5_FLASH_LITE}'. Check API Key and model name. Error: {e}")

Agent 'weather_agent_gemini' created using model 'gemini/gemini-2.5-flash-lite'.
Session created: App='weather_tutorial_app_gemini', User='user_1_gemini', Session='session_001_gemini'
Runner created for agent 'weather_agent_gemini'.

--- Testing gemini Agent ---

>>> User Query: What's the weather in Tokyo?
  [Event] Author: weather_agent_gemini, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'city': 'Tokyo'
    },
    id='call_93ff50527cad4ef4b7da0d1cf665',
    name='get_weather'
  )
)] role='model'
--- Tool: get_weather called for city: Tokyo ---
  [Event] Author: weather_agent_gemini, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='call_93ff50527cad4ef4b7da0d1cf665',
    name='get_weather',
    response={
      'report': 'Tokyo is experiencing light rain and a temperature of 18°C.',
      'status': 'success'
    }
  )
)] role='user'
  [Event] Author: weather_agent_gemini, Type: Event, Fin

Definindo e Testando o Agente Groq, era para ser o do Claude nessa etapa, mas novamente ele exige pagamento para o uso de algum de seus modelos, então foi utilizado o groq

In [ ]:
# --- Agent usando o Groq ---
weather_agent_groq = None # Inicializa a variável do agente como None
runner_groq = None        # Inicializa a variável do runner como None

try:
    weather_agent_groq = Agent(
        name="weather_agent_groq", # Define um nome para o Agente
        model=LiteLlm(model=MODEL_GROQ), # Configura o agente para usar um modelo via LiteLLM
        description="Provides weather information (using Groq).", # Descrição curta do que o agente faz
        instruction="You are a helpful weather assistant powered by Groq. " # Instrução para orientar o agente
                    "Use the 'get_weather' tool for city weather requests. "
                    "Analyze the tool's dictionary output ('status', 'report'/'error_message'). "
                    "Clearly present successful reports or polite error messages.",
        tools=[get_weather], # Usa novamente a mesma ferramenta
    )
    print(f"Agent '{weather_agent_groq.name}' created using model '{MODEL_GROQ}'.")

    # InMemorySessionService simples, não salva em banco.
    session_service_groq = InMemorySessionService() # Cria um servidor dedicado

    APP_NAME_GROQ = "weather_tutorial_app_groq" # Nome da aplicação
    USER_ID_GROQ = "user_1_groq" # Id do usuário, para distinguir sessões de usuários diferentes
    SESSION_ID_GROQ = "session_001_groq" # Identificador fixo de sessão

    # Cria a sessão específica a conversa irá ocorrer
    session_groq = await session_service_groq.create_session(
        app_name=APP_NAME_GROQ,
        user_id=USER_ID_GROQ,
        session_id=SESSION_ID_GROQ
    )
    print(f"Session created: App='{APP_NAME_GROQ}', User='{USER_ID_GROQ}', Session='{SESSION_ID_GROQ}'")

    # Agora cria o Runner específico desse agente e desse serviço de sessão
    runner_groq = Runner( # Instancia o Runner que vai executar o agente em loop de eventos.
        agent=weather_agent_groq,
        app_name=APP_NAME_GROQ,       # Usa o nome da aplicação
        session_service=session_service_groq # Usa a sessão específica
        )
    print(f"Runner created for agent '{runner_groq.agent.name}'.")

    # --- Testa o Agente Groq ---
    print("\n--- Testing Groq Agent ---")
    await call_agent_async(query = "Weather in London please.",
                           runner=runner_groq,
                           user_id=USER_ID_GROQ,
                           session_id=SESSION_ID_GROQ)

except Exception as e:
    print(f"❌ Could not create or run Claude agent '{MODEL_GROQ}'. Check API Key and model name. Error: {e}")

Agent 'weather_agent_groq' created using model 'groq/llama-3.3-70b-versatile'.
Session created: App='weather_tutorial_app_groq', User='user_1_groq', Session='session_001_groq'
Runner created for agent 'weather_agent_groq'.

--- Testing Groq Agent ---

>>> User Query: Weather in London please.
  [Event] Author: weather_agent_groq, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'city': 'London'
    },
    id='n6frjqn17',
    name='get_weather'
  )
)] role='model'
--- Tool: get_weather called for city: London ---
  [Event] Author: weather_agent_groq, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='n6frjqn17',
    name='get_weather',
    response={
      'report': "It's cloudy in London with a temperature of 15°C.",
      'status': 'success'
    }
  )
)] role='user'
  [Event] Author: weather_agent_groq, Type: Event, Final: True, Content: parts=[Part(
  text='The current weather in London is clou

Definindo Ferramentas para os Agentes de Boas-vindas e despedidas

In [ ]:
from typing import Optional

def say_hello(name: Optional[str] = None) -> str:
    """Provides a simple greeting. If a name is provided, it will be used.

    Args:
        name (str, optional): The name of the person to greet. Defaults to a generic greeting if not provided.

    Returns:
        str: A friendly greeting message.
    """
    if name: # Verifica se name possui um valor considerado verdadeiro
        greeting = f"Hello, {name}!"
        print(f"--- Tool: say_hello called with name: {name} ---")
    else: # Executado quando name não foi fornecido ou é None
        greeting = "Hello there!"
        print(f"--- Tool: say_hello called without a specific name (name_arg_value: {name}) ---")
    return greeting

def say_goodbye() -> str: # Uma simples função para retornar uma despedida
    """Provides a simple farewell message to conclude the conversation."""
    print(f"--- Tool: say_goodbye called ---")
    return "Goodbye! Have a great day."

print("Greeting and Farewell tools defined.")

# Teste
print(say_hello("Alice"))
print(say_hello()) # Teste sem argumento
print(say_hello(name=None)) # Teste com 'name' sendo definido como None explicitamente

Greeting and Farewell tools defined.
--- Tool: say_hello called with name: Alice ---
Hello, Alice!
--- Tool: say_hello called without a specific name (name_arg_value: None) ---
Hello there!
--- Tool: say_hello called without a specific name (name_arg_value: None) ---
Hello there!


Definindo os sub-agentes de boas-vindas e de despedida

In [ ]:
# --- Agente de Boas-vindas ---
greeting_agent = None # Inicializa a variável do agente como None
try:
    greeting_agent = Agent(
        model = MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent", # Define um nome para o Agente
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting to the user. " # Instrução para orientar o agente
                    "Use the 'say_hello' tool to generate the greeting. "
                    "If the user provides their name, make sure to pass it to the tool. "
                    "Do not engage in any other conversation or tasks.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.", # Descrição curta do que o agente faz, mas crucial para a delegação
        tools=[say_hello], # Lista das ferramentas disponíveis para uso
    )
    print(f"✅ Agent '{greeting_agent.name}' created using model '{greeting_agent.model}'.")
except Exception as e:
    print(f"❌ Could not create Greeting agent. Check API Key ({greeting_agent.model}). Error: {e}")

# --- Agente de despedida ---
farewell_agent = None # Inicializa a variável do agente como None
try:
    farewell_agent = Agent(
        model = MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent", # Define um nome para o Agente
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message. " # Instrução para orientar o agente
                    "Use the 'say_goodbye' tool when the user indicates they are leaving or ending the conversation "
                    "(e.g., using words like 'bye', 'goodbye', 'thanks bye', 'see you'). "
                    "Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.", # Descrição curta do que o agente faz, mas crucial para a delegação
        tools=[say_goodbye], # Lista das ferramentas disponíveis para uso
    )
    print(f"✅ Agent '{farewell_agent.name}' created using model '{farewell_agent.model}'.")
except Exception as e:
    print(f"❌ Could not create Farewell agent. Check API Key ({farewell_agent.model}). Error: {e}")

✅ Agent 'greeting_agent' created using model 'gemini-2.5-flash'.
✅ Agent 'farewell_agent' created using model 'gemini-2.5-flash'.


Definindo o Agente Raiz com os sub-agentes

In [ ]:
root_agent = None # Inicializa a variável do agente raiz como None
runner_root = None # Inicializa a variável do runner do agente raiz como None

if greeting_agent and farewell_agent and 'get_weather' in globals(): # Verifica se os dois sub-agentes existem e se a função get_weather está definida no escopo global.
    root_agent_model = MODEL_GEMINI_2_5_FLASH

    weather_agent_team = Agent(
        name="weather_agent_v2", # Define um nome para o Agente
        model=root_agent_model,
        description="The main coordinator agent. Handles weather requests and delegates greetings/farewells to specialists.", # Descrição curta do que o agente faz, nesse caso ele irá coordenar o time
        instruction="You are the main Weather Agent coordinating a team. Your primary responsibility is to provide weather information. " # Instrução para orientar o agente
                    "Use the 'get_weather' tool ONLY for specific weather requests (e.g., 'weather in London'). "
                    "You have specialized sub-agents: "
                    "1. 'greeting_agent': Handles simple greetings like 'Hi', 'Hello'. Delegate to it for these. "
                    "2. 'farewell_agent': Handles simple farewells like 'Bye', 'See you'. Delegate to it for these. "
                    "Analyze the user's query. If it's a greeting, delegate to 'greeting_agent'. If it's a farewell, delegate to 'farewell_agent'. "
                    "If it's a weather request, handle it yourself using 'get_weather'. "
                    "For anything else, respond appropriately or state you cannot handle it.",
        tools=[get_weather], # Registra a tool get_weather para que o agente raiz possa executar sua função principal
        sub_agents=[greeting_agent, farewell_agent] # Deixa claro quais agentes são sub-agentes delegáveis
    )
    print(f"✅ Root Agent '{weather_agent_team.name}' created using model '{root_agent_model}' with sub-agents: {[sa.name for sa in weather_agent_team.sub_agents]}")

else:
    print("❌ Cannot create root agent because one or more sub-agents failed to initialize or 'get_weather' tool is missing.")
    if not greeting_agent: print(" - Greeting Agent is missing.")
    if not farewell_agent: print(" - Farewell Agent is missing.")
    if 'get_weather' not in globals(): print(" - get_weather function is missing.")

✅ Root Agent 'weather_agent_v2' created using model 'gemini-2.5-flash' with sub-agents: ['greeting_agent', 'farewell_agent']


Interagindo com o time de agentes

In [ ]:
import asyncio

root_agent_var_name = 'root_agent' # Define o nome padrão da variável
if 'weather_agent_team' in globals(): # Verifica se existe uma variável global chamada weather_agent_team
    root_agent_var_name = 'weather_agent_team' # Se existir, atualiza o nome da variável “fonte” do root agent para weather_agent_team
elif 'root_agent' not in globals(): # Se weather_agent_team não existe e também não existe root_agent, então o agente raiz não está disponível
    print("⚠️ Root agent ('root_agent' or 'weather_agent_team') not found. Cannot define run_team_conversation.")
    root_agent = None # Define root_agent como None para evitar NameError caso alguma parte referencie root_agent depois.

if root_agent_var_name in globals() and globals()[root_agent_var_name]: # Checa se o nome da variável está no escopo global e se essa variável não é None
    async def run_team_conversation():
        print("\n--- Testing Agent Team Delegation ---")
        session_service = InMemorySessionService() # Cria um serviço de sessão em memória para armazenar o histórico
        APP_NAME = "weather_tutorial_agent_team" # Define o nome da aplicação
        USER_ID = "user_1_agent_team" # Define o ID do usuário específico desse teste
        SESSION_ID = "session_001_agent_team" # Define o ID da sessão específica desse teste
        session = await session_service.create_session( # Cria a sessão onde a conversa acontecerá
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )
        print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

        actual_root_agent = globals()[root_agent_var_name] # Busca o objeto do root agent diretamente do dicionário global
        runner_agent_team = Runner( # Cria um Runner que vai executar o root agent
            agent=actual_root_agent,
            app_name=APP_NAME,
            session_service=session_service
        )
        print(f"Runner created for agent '{actual_root_agent.name}'.")

        # Interações
        await call_agent_async(query = "Hello there!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "What is the weather in New York?",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)
        await call_agent_async(query = "Thanks, bye!",
                               runner=runner_agent_team,
                               user_id=USER_ID,
                               session_id=SESSION_ID)

    # Execução
    print("Attempting execution using 'await' (default for notebooks)...")
    await run_team_conversation()

else: # Caso o root agent não exista/esteja None
    print("\n⚠️ Skipping agent team conversation execution as the root agent was not successfully defined in a previous step.")

Attempting execution using 'await' (default for notebooks)...

--- Testing Agent Team Delegation ---
Session created: App='weather_tutorial_agent_team', User='user_1_agent_team', Session='session_001_agent_team'
Runner created for agent 'weather_agent_v2'.

>>> User Query: Hello there!
  [Event] Author: weather_agent_v2, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'agent_name': 'greeting_agent'
    },
    id='adk-08ca8a88-17c0-43aa-a545-f335b917b739',
    name='transfer_to_agent'
  ),
  thought_signature=b'\n\x81\x02\x01\xbe>\xf6\xfb\xd1a\xaaB\xd0\x80\xa2\x0f\xfb\x8cjs\x12D\xdfV\x9b\x8f\x85\xde=\xd3\x19\x1b\xb3.\xd3\x9d\x1b\n*SnQ\x1f\x11\x1al!0\x96\x94\xd9\x8e\x1e+\x9d\x98o\t.^\xe8\x8cW\xd3m\xcdS\xea@p\xfbXsr\xb0\xcarF9i\xbee}ZADA\x02\xe5\xbbH\x1bmI\x9b-d...'
)] role='model'
  [Event] Author: weather_agent_v2, Type: Event, Final: False, Content: parts=[Part(
  function_response=FunctionResponse(
    id='adk-08ca8a88-17c0-43aa-a545-f33

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 54.660910697s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_

Iniciando um novo serviço de sessão e estado

In [ ]:
from google.adk.sessions import InMemorySessionService

# Criando uma nova instância de serviço de sessão para demonstrar estado
session_service_stateful = InMemorySessionService() # Cria uma nova instância do serviço de sessão
print("✅ New InMemorySessionService created for state demonstration.")

# Define um novo ID de sessão e de usuário para essa parte
SESSION_ID_STATEFUL = "session_state_demo_001"
USER_ID_STATEFUL = "user_state_demo"

# Define os dados do estado inicial - O usuário prefere Celsius
initial_state = {
    "user_preference_temperature_unit": "Celsius"
}

# Cria a sessão, indicando o estado inicial
session_stateful = await session_service_stateful.create_session(
    app_name=APP_NAME, # Associa a sessão ao mesmo nome de aplicação usado anteriormente
    user_id=USER_ID_STATEFUL, # Associa a sessão ao usuário específico dessa demonstração
    session_id=SESSION_ID_STATEFUL, # Define o identificador único da sessão
    state=initial_state # Inicializa o estado durante a criação
)
print(f"✅ Session '{SESSION_ID_STATEFUL}' created for user '{USER_ID_STATEFUL}'.")

# Verifica se o estado inicial está configurado corretamente
retrieved_session = await session_service_stateful.get_session(app_name=APP_NAME,
                                                         user_id=USER_ID_STATEFUL,
                                                         session_id = SESSION_ID_STATEFUL)
print("\n--- Initial Session State ---")
if retrieved_session:
    print(retrieved_session.state)
else:
    print("Error: Could not retrieve session.")

✅ New InMemorySessionService created for state demonstration.
✅ Session 'session_state_demo_001' created for user 'user_state_demo'.

--- Initial Session State ---
{'user_preference_temperature_unit': 'Celsius'}


In [ ]:
from google.adk.tools.tool_context import ToolContext

def get_weather_stateful(city: str, tool_context: ToolContext) -> dict:
    """Retrieves weather, converts temp unit based on session state."""
    print(f"--- Tool: get_weather_stateful called for {city} ---")

    preferred_unit = tool_context.state.get("user_preference_temperature_unit", "Celsius") # Lê do estado da sessão a preferência de unidade de temperatura; se não existir, usa "Celsius" como padrão
    print(f"--- Tool: Reading state 'user_preference_temperature_unit': {preferred_unit} ---")

    city_normalized = city.lower().replace(" ", "") # Normaliza o nome da cidade

    mock_weather_db = { # Banco de dados simulado
        "newyork": {"temp_c": 25, "condition": "sunny"},
        "london": {"temp_c": 15, "condition": "cloudy"},
        "tokyo": {"temp_c": 18, "condition": "light rain"},
    }

    if city_normalized in mock_weather_db: # Verifica se a cidade normalizada existe no banco de dados simulado
        data = mock_weather_db[city_normalized] # Recupera os dados climáticos da cidade
        temp_c = data["temp_c"] # Extrai a temperatura
        condition = data["condition"] # Extrai a condição

        # Formata a temparuta dependendo da preferência de estado
        if preferred_unit == "Fahrenheit":
            temp_value = (temp_c * 9/5) + 32 # Converte Celsius para Fahrenheit
            temp_unit = "°F"
        else: # Padrão é ser Celsius
            temp_value = temp_c
            temp_unit = "°C"

        report = f"The weather in {city.capitalize()} is {condition} with a temperature of {temp_value:.0f}{temp_unit}."
        result = {"status": "success", "report": report}
        print(f"--- Tool: Generated report in {preferred_unit}. Result: {result} ---")

        tool_context.state["last_city_checked_stateful"] = city # Escreve no estado da sessão a última cidade consultada
        print(f"--- Tool: Updated state 'last_city_checked_stateful': {city} ---")

        return result
    else: # Caso a cidade não seja encontrada
        error_msg = f"Sorry, I don't have weather information for '{city}'."
        print(f"--- Tool: City '{city}' not found. ---")
        return {"status": "error", "error_message": error_msg}

print("✅ State-aware 'get_weather_stateful' tool defined.")

✅ State-aware 'get_weather_stateful' tool defined.


Redefinindo os sub-agentes e atualizando o agente raiz com output_key

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner

# Redefinindo o agente de boas vindas
greeting_agent = None
try:
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent",
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"✅ Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Greeting agent. Error: {e}")

# Redefinindo o agente de depesdida
farewell_agent = None
try:
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent",
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"✅ Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Farewell agent. Error: {e}")

# Definindo o Agente raiz atualizado
root_agent_stateful = None # Inicializa a variável do agente raiz como None
runner_root_stateful = None # Inicializa a variável do runner do agente raiz como None

# Verifica os pré-requisitos antes da criação do agente raiz
if greeting_agent and farewell_agent and 'get_weather_stateful' in globals():

    root_agent_model = MODEL_GEMINI_2_5_FLASH # Modelo que será o orquestrador

    root_agent_stateful = Agent(
        name="weather_agent_v4_stateful", # Define o nome do agente raiz
        model=root_agent_model,
        description="Main agent: Provides weather (state-aware unit), delegates greetings/farewells, saves report to state.", # Descreve o papel do agente raiz, incluindo uso de estado e delegação.
        instruction="You are the main Weather Agent. Your job is to provide weather using 'get_weather_stateful'. " # Instruções que definem o comportamento principal do agente raiz
                    "The tool will format the temperature based on user preference stored in state. "
                    "Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                    "Handle only weather requests, greetings, and farewells.",
        tools=[get_weather_stateful], # Registra a tool de clima que lê e escreve no estado da sessão
        sub_agents=[greeting_agent, farewell_agent], # Inclue os sub-agentes
        output_key="last_weather_report" # Define a chave do estado onde a resposta final de clima será salva automaticamente
    )
    print(f"✅ Root Agent '{root_agent_stateful.name}' created using stateful tool and output_key.")

    # Criação do runner
    runner_root_stateful = Runner( # Cria o Runner responsável por executar o agente raiz com estado
        agent=root_agent_stateful,
        app_name=APP_NAME,
        session_service=session_service_stateful # Usa o serviço de sessão com estado (onde preferências e respostas são armazenadas)
    )
    print(f"✅ Runner created for stateful root agent '{runner_root_stateful.agent.name}' using stateful session service.")

else:
    print("❌ Cannot create stateful root agent. Prerequisites missing.")
    if not greeting_agent: print(" - greeting_agent definition missing.")
    if not farewell_agent: print(" - farewell_agent definition missing.")
    if 'get_weather_stateful' not in globals(): print(" - get_weather_stateful tool missing.")

✅ Agent 'greeting_agent' redefined.
✅ Agent 'farewell_agent' redefined.
✅ Root Agent 'weather_agent_v4_stateful' created using stateful tool and output_key.
✅ Runner created for stateful root agent 'weather_agent_v4_stateful' using stateful session service.


Testando o fluxo da conversa e o output_key

In [ ]:
import asyncio

if 'runner_root_stateful' in globals() and runner_root_stateful: # Verifica se a variável global runner_root_stateful existe e se ela não é None
    async def run_stateful_conversation(): # Declara a função assíncrona que vai executar os “turnos” da conversa e testar o estado
        print("\n--- Testing State: Temp Unit Conversion & output_key ---")

        # 1. Verificar clima (Usa o estado inicial: Celsius)
        print("--- Turn 1: Requesting weather in London (expect Celsius) ---")
        await call_agent_async(query= "What's the weather in London?", # Envia a pergunta ao agente
                               runner=runner_root_stateful,
                               user_id=USER_ID_STATEFUL,
                               session_id=SESSION_ID_STATEFUL
                              )

        # 2. A preferência será alterada manualmente para Fahrenheit (mexendo diretamente no storage em memória)
        print("\n--- Manually Updating State: Setting unit to Fahrenheit ---")
        try:
            # Acessar sessions[...] é algo específico do InMemorySessionService e serve só para testes
            stored_session = session_service_stateful.sessions[APP_NAME][USER_ID_STATEFUL][SESSION_ID_STATEFUL] # Acessa diretamente a sessão guardada dentro do dicionário interno do serviço em memória
            stored_session.state["user_preference_temperature_unit"] = "Fahrenheit" # Atualiza a preferência no state para Fahrenheit
            print(f"--- Stored session state updated. Current 'user_preference_temperature_unit': {stored_session.state.get('user_preference_temperature_unit', 'Not Set')} ---") # Added .get for safety
        except KeyError:
            print(f"--- Error: Could not retrieve session '{SESSION_ID_STATEFUL}' from internal storage for user '{USER_ID_STATEFUL}' in app '{APP_NAME}' to update state. Check IDs and if session was created. ---")
        except Exception as e:
             print(f"--- Error updating internal session state: {e} ---")

        # 3. Verifica clima novament (Deve ser usado Fahrenheit agora)
        # Isso também irá atualizar 'last_weather_report' via output_key
        print("\n--- Turn 2: Requesting weather in New York (expect Fahrenheit) ---")
        await call_agent_async(query= "Tell me the weather in New York.", # Envia o pedido de clima em New York
                               runner=runner_root_stateful,
                               user_id=USER_ID_STATEFUL,
                               session_id=SESSION_ID_STATEFUL
                              )

        # 4. Testa delegação
        # Isso irá atualizar 'last_weather_report' novamente, sobrescrevendo o reporte de clima de NY
        print("\n--- Turn 3: Sending a greeting ---")
        await call_agent_async(query= "Hi!", # Envia uma saudação; o root agent deve delegar ao greeting_agent
                               runner=runner_root_stateful,
                               user_id=USER_ID_STATEFUL,
                               session_id=SESSION_ID_STATEFUL
                              )

    print("Attempting execution using 'await' (default for notebooks)...")
    await run_stateful_conversation() # Executa toda a sequência de testes


    # Inspeciona o estado da sessão final depois da conversa
    # Este bloco é executado após a conclusão de qualquer um dos métodos de execução
    print("\n--- Inspecting Final Session State ---")
    final_session = await session_service_stateful.get_session(app_name=APP_NAME, # Recupera a sessão do serviço de sessão
                                                         user_id= USER_ID_STATEFUL,
                                                         session_id=SESSION_ID_STATEFUL)
    if final_session: # Verifica se a sessão foi recuperada com sucesso
        # Usa .get() para evitar erro se alguma chave não existir
        print(f"Final Preference: {final_session.state.get('user_preference_temperature_unit', 'Not Set')}")
        print(f"Final Last Weather Report (from output_key): {final_session.state.get('last_weather_report', 'Not Set')}")
        print(f"Final Last City Checked (by tool): {final_session.state.get('last_city_checked_stateful', 'Not Set')}")
    else:
        print("\n❌ Error: Could not retrieve final session state.")

else:
    print("\n⚠️ Skipping state test conversation. Stateful root agent runner ('runner_root_stateful') is not available.")

Attempting execution using 'await' (default for notebooks)...

--- Testing State: Temp Unit Conversion & output_key ---
--- Turn 1: Requesting weather in London (expect Celsius) ---

>>> User Query: What's the weather in London?
  [Event] Author: weather_agent_v4_stateful, Type: Event, Final: False, Content: parts=[Part(
  function_call=FunctionCall(
    args={
      'city': 'London'
    },
    id='adk-a745affd-8570-42fc-9efe-3a47c0a30fd0',
    name='get_weather_stateful'
  ),
  thought_signature=b'\n\xc3\x02\x01\xbe>\xf6\xfb"\xe6\xc8P\x1aq\xe6BpO\x08\xb3\xb3\xed\xba\xb0\t\r\xeb\xf0\x99\x88\x0b\xdd9\x10\xebfh\xf0\x11\x02\x01F7\xe2F\xe1\xe0\x07K\xd3\x801upq\x8f\xd2G\x0f\x19\xd1\xcb\xf2s\xf3\x8b]\x81^\xcf\na\xb1\x88\x8d\xe7\xc7\t\xfd\xe7JZ?m\x18\xf6rZ9\xe4|\xcc\xe9\xa598~...'
)] role='model'
--- Tool: get_weather_stateful called for London ---
--- Tool: Reading state 'user_preference_temperature_unit': Celsius ---
--- Tool: Generated report in Celsius. Result: {'status': 'success', 'repo

_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 46.190229565s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier

Define o before_model_callback que funciona como barreira de segurança

In [ ]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.genai import types
from typing import Optional

def block_keyword_guardrail( # Define a função de callback que atuará como guardrail
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]: # Define que a função pode retornar um LlmResponse (bloqueando a chamada) ou None (permitindo a continuação)

    agent_name = callback_context.agent_name # Obtém o nome do agente cujo chamado ao modelo está sendo interceptado
    print(f"--- Callback: block_keyword_guardrail running for agent: {agent_name} ---")

    # Extrai o texto da última mensagem do usuário no histórico de requisição
    last_user_message_text = ""
    if llm_request.contents: # Verifica se a requisição contém histórico de mensagens
        for content in reversed(llm_request.contents): # Itera sobre as mensagens da requisição de trás para frente
            if content.role == 'user' and content.parts: # Verifica se a mensagem é do usuário e se possui partes de conteúdo
                if content.parts[0].text: # Confirma que a primeira parte contém texto
                    last_user_message_text = content.parts[0].text # Armazena o texto da última mensagem do usuário
                    break # Interrompe o loop após encontrar a mensagem de usuário mais recente

    print(f"--- Callback: Inspecting last user message: '{last_user_message_text[:100]}...' ---")

    # Lógica Barreira de proteção
    keyword_to_block = "BLOCK" # Define a palavra-chave que deve acionar o bloqueio
    if keyword_to_block in last_user_message_text.upper(): # Verifica se a palavra “BLOCK” aparece na mensagem
        print(f"--- Callback: Found '{keyword_to_block}'. Blocking LLM call! ---")
        callback_context.state["guardrail_block_keyword_triggered"] = True # Escreve no estado da sessão que o guardrail foi acionado
        print(f"--- Callback: Set state 'guardrail_block_keyword_triggered': True ---")

        # Construindo e retornando um LlmResponse para interromper o fluxo
        return LlmResponse(
            content=types.Content(
                role="model", # Mimic a response from the agent's perspective
                parts=[types.Part(text=f"I cannot process this request because it contains the blocked keyword '{keyword_to_block}'.")], # Cria o texto da resposta informando que a requisição foi bloqueada pela palavra-chave
            )

        )
    else: # Executado quando a palavra bloqueada não é encontrada
        print(f"--- Callback: Keyword not found. Allowing LLM call for {agent_name}. ---")
        return None # sinaliza ao ADK que o processamento deve continuar normalmente

print("✅ block_keyword_guardrail function defined.")

✅ block_keyword_guardrail function defined.


Atualizando o Agenre raiz com o before_model_callback

In [ ]:
# Redefinindo os sub-agentes
greeting_agent = None
try:
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent", # Mantendo o nome original para consistência
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"✅ Sub-Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Greeting agent. Check Model/API Key ({greeting_agent.model}). Error: {e}")

farewell_agent = None
try:
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent", # Mantendo o nome original
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"✅ Sub-Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Farewell agent. Check Model/API Key ({farewell_agent.model}). Error: {e}")


# Definindo o agente raiz com o callback
root_agent_model_guardrail = None
runner_root_model_guardrail = None

if greeting_agent and farewell_agent and 'get_weather_stateful' in globals() and 'block_keyword_guardrail' in globals(): # Só continua se os sub-agentes existem, a tool stateful existe e o callback guardrail existe

    root_agent_model = MODEL_GEMINI_2_5_FLASH

    root_agent_model_guardrail = Agent(
        name="weather_agent_v5_model_guardrail", # Nova versão do nome para clareza
        model=root_agent_model,
        description="Main agent: Handles weather, delegates greetings/farewells, includes input keyword guardrail.",
        instruction="You are the main Weather Agent. Provide weather using 'get_weather_stateful'. "
                    "Delegate simple greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                    "Handle only weather requests, greetings, and farewells.",
        tools=[get_weather_stateful], # Registra a tool de clima com estado
        sub_agents=[greeting_agent, farewell_agent], # Conecta os sub-agentes redefinidos ao root agent para delegação
        output_key="last_weather_report", # Configura a chave do state onde a resposta final do turno será salva automaticamente
        before_model_callback=block_keyword_guardrail # Registra o callback que roda antes de chamar o modelo
    )
    print(f"✅ Root Agent '{root_agent_model_guardrail.name}' created with before_model_callback.")

    # Criando um Runner usando o mesmo serviço de sessão com estado
    if 'session_service_stateful' in globals(): # Verifica se o serviço de sessão com estado existe no escopo global
        runner_root_model_guardrail = Runner(
            agent=root_agent_model_guardrail,
            app_name=APP_NAME, # Associa o Runner ao nome da aplicação
            session_service=session_service_stateful # Usa o serviço de sessão com estado
        )
        print(f"✅ Runner created for guardrail agent '{runner_root_model_guardrail.agent.name}', using stateful session service.")
    else:
        print("❌ Cannot create runner. 'session_service_stateful' from Step 4 is missing.")

else: # Executado se algum pré-requisito (sub-agentes, tool, callback) estiver faltando
    print("❌ Cannot create root agent with model guardrail. One or more prerequisites are missing or failed initialization:")
    if not greeting_agent: print("   - Greeting Agent")
    if not farewell_agent: print("   - Farewell Agent")
    if 'get_weather_stateful' not in globals(): print("   - 'get_weather_stateful' tool")
    if 'block_keyword_guardrail' not in globals(): print("   - 'block_keyword_guardrail' callback")

✅ Sub-Agent 'greeting_agent' redefined.
✅ Sub-Agent 'farewell_agent' redefined.
✅ Root Agent 'weather_agent_v5_model_guardrail' created with before_model_callback.
✅ Runner created for guardrail agent 'weather_agent_v5_model_guardrail', using stateful session service.


Testando a entrada do modelo com Guardrail

In [ ]:
import asyncio

if 'runner_root_model_guardrail' in globals() and runner_root_model_guardrail: # Verifica se o runner do root agent com guardrail existe no escopo global e está válido
    async def run_guardrail_test_conversation(): # Define uma função assíncrona que executa o teste do guardrail
        print("\n--- Testing Model Input Guardrail ---")

        interaction_func = lambda query: call_agent_async(query, # Cria uma função lambda que recebe query e chama call_agent_async
                                                         runner_root_model_guardrail, # Passa o runner do agente com guardrail
                                                         USER_ID_STATEFUL, # Usa o ID do usuário existente
                                                         SESSION_ID_STATEFUL # Usa o ID da sessão existente
                                                        )
        # 1. Primeira interação é normal; deve passar no guardrail e usar Fahrenheit (se o state já foi alterado antes)
        print("--- Turn 1: Requesting weather in London (expect allowed, Fahrenheit) ---")
        await interaction_func("What is the weather in London?") # Envia a pergunta; o callback deve permitir e o agente deve responder o clima

        # 2. Segunda interação contém a palavra bloqueada, então o callback deve interceptar
        print("\n--- Turn 2: Requesting with blocked keyword (expect blocked) ---")
        await interaction_func("BLOCK the request for weather in Tokyo") # Envia uma query com “BLOCK”; o callback deve detectar e impedir a chamada ao modelo

        # 3. Terceira interação é uma saudação; deve passar no guardrail e o root agent deve delegar ao greeting_agent
        print("\n--- Turn 3: Sending a greeting (expect allowed) ---")
        await interaction_func("Hello again") # Envia a saudação; o root agent deve delegar para o sub-agente de greeting


    print("Attempting execution using 'await' (default for notebooks)...")
    await run_guardrail_test_conversation() # Executa a função assíncrona, rodando todos os turnos do teste

    # Inspecionando o estado final da sessão após o teste
    print("\n--- Inspecting Final Session State (After Guardrail Test) ---")
    final_session = await session_service_stateful.get_session(app_name=APP_NAME, # Recupera a sessão para ler o estado
                                                         user_id=USER_ID_STATEFUL,
                                                         session_id=SESSION_ID_STATEFUL)
    if final_session: # Verifica se a sessão foi recuperada com sucesso
        print(f"Guardrail Triggered Flag: {final_session.state.get('guardrail_block_keyword_triggered', 'Not Set (or False)')}") # Imprime a flag que o callback setou quando encontrou “BLOCK” (deve ser True após o Turno 2)
        print(f"Last Weather Report: {final_session.state.get('last_weather_report', 'Not Set')}") # Deve ser o clima de Londres se bem sucedido
        print(f"Temperature Unit: {final_session.state.get('user_preference_temperature_unit', 'Not Set')}") # Deve ser Fahrenheit
    else:
        print("\n❌ Error: Could not retrieve final session state.")

else:
    print("\n⚠️ Skipping model guardrail test. Runner ('runner_root_model_guardrail') is not available.")

Attempting execution using 'await' (default for notebooks)...

--- Testing Model Input Guardrail ---
--- Turn 1: Requesting weather in London (expect allowed, Fahrenheit) ---

>>> User Query: What is the weather in London?
--- Callback: block_keyword_guardrail running for agent: weather_agent_v5_model_guardrail ---
--- Callback: Inspecting last user message: 'What is the weather in London?...' ---
--- Callback: Keyword not found. Allowing LLM call for weather_agent_v5_model_guardrail. ---


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 50.190341183s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier

Definindo a barrieira de proteção before_tool_callback

In [ ]:
from google.adk.tools.base_tool import BaseTool
from google.adk.tools.tool_context import ToolContext
from typing import Optional, Dict, Any

def block_paris_tool_guardrail( # Define a função de callback que atuará como guardrail antes da execução da tool
    tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext # Passa a tool que está para ser executado, seus argumentos e o contexto da execução
) -> Optional[Dict]: # Define que a função pode retornar um dicionário (bloqueando a tool) ou None (permitindo a execução)
    """
    Checks if 'get_weather_stateful' is called for 'Paris'.
    If so, blocks the tool execution and returns a specific error dictionary.
    Otherwise, allows the tool call to proceed by returning None.
    """
    tool_name = tool.name # Obtém o nome da tool que está sendo chamada
    agent_name = tool_context.agent_name # Obtém o nome do agente que tentou executar a tool
    print(f"--- Callback: block_paris_tool_guardrail running for tool '{tool_name}' in agent '{agent_name}' ---")
    print(f"--- Callback: Inspecting args: {args} ---")

    # Lógica da Barreira de proteção
    target_tool_name = "get_weather_stateful" # Define o nome da tool que deve ser bloqueada
    blocked_city = "paris" # Define a cidade que será bloqueada

    # Verifica se é a ferramenta correta e se é o argumento da cidade correspondente à cidade bloqueada
    if tool_name == target_tool_name: # Verifica se a tool chamada é exatamente a tool alvo
        city_argument = args.get("city", "") # Obtém o argumento city de forma segura, se não existir usa string vazia
        if city_argument and city_argument.lower() == blocked_city: # Verifica se a cidade foi fornecida e se é igual a “paris”
            print(f"--- Callback: Detected blocked city '{city_argument}'. Blocking tool execution! ---")
            tool_context.state["guardrail_tool_block_triggered"] = True # Salva no estado da sessão que o guardrail de tool foi acionado
            print(f"--- Callback: Set state 'guardrail_tool_block_triggered': True ---")

            return {
                "status": "error",
                "error_message": f"Policy restriction: Weather checks for '{city_argument.capitalize()}' are currently disabled by a tool guardrail."
            }
        else:
             print(f"--- Callback: City '{city_argument}' is allowed for tool '{tool_name}'. ---")
    else:
        print(f"--- Callback: Tool '{tool_name}' is not the target tool. Allowing. ---")


    # Se a vericação acima não retornou um dicionário, permiti a ferramenta executar
    print(f"--- Callback: Allowing tool '{tool_name}' to proceed. ---")
    return None # Sinaliza ao ADK que a tool original deve ser executada

print("✅ block_paris_tool_guardrail function defined.")

✅ block_paris_tool_guardrail function defined.


Atualizando o agente raiz com ambos os callbacks

In [ ]:
# Redefinindo os sub-agentes
greeting_agent = None
try:
    greeting_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="greeting_agent", # Mantém o nome original para consistência
        instruction="You are the Greeting Agent. Your ONLY task is to provide a friendly greeting using the 'say_hello' tool. Do nothing else.",
        description="Handles simple greetings and hellos using the 'say_hello' tool.",
        tools=[say_hello],
    )
    print(f"✅ Sub-Agent '{greeting_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Greeting agent. Check Model/API Key ({greeting_agent.model}). Error: {e}")

farewell_agent = None
try:
    farewell_agent = Agent(
        model=MODEL_GEMINI_2_5_FLASH,
        name="farewell_agent", # Mantém o nome original
        instruction="You are the Farewell Agent. Your ONLY task is to provide a polite goodbye message using the 'say_goodbye' tool. Do not perform any other actions.",
        description="Handles simple farewells and goodbyes using the 'say_goodbye' tool.",
        tools=[say_goodbye],
    )
    print(f"✅ Sub-Agent '{farewell_agent.name}' redefined.")
except Exception as e:
    print(f"❌ Could not redefine Farewell agent. Check Model/API Key ({farewell_agent.model}). Error: {e}")

# Definindo o Agente raiz com ambos os callbacks
root_agent_tool_guardrail = None
runner_root_tool_guardrail = None

if ('greeting_agent' in globals() and greeting_agent and # Verifica se a variável greeting_agent existe globalmente e se ela contém um objeto válido
    'farewell_agent' in globals() and farewell_agent and # Verifica o mesmo para o farewell_agent
    'get_weather_stateful' in globals() and # Verifica se a tool get_weather_stateful foi definida
    'block_keyword_guardrail' in globals() and # Verifica se o callback de guardrail antes do modelo existe
    'block_paris_tool_guardrail' in globals()): # Verifica se o callback de guardrail antes da tool existe

    root_agent_model = MODEL_GEMINI_2_5_FLASH

    root_agent_tool_guardrail = Agent(
        name="weather_agent_v6_tool_guardrail", # Nova versão do nome
        model=root_agent_model,
        description="Main agent: Handles weather, delegates, includes input AND tool guardrails.",
        instruction="You are the main Weather Agent. Provide weather using 'get_weather_stateful'. "
                    "Delegate greetings to 'greeting_agent' and farewells to 'farewell_agent'. "
                    "Handle only weather, greetings, and farewells.",
        tools=[get_weather_stateful],
        sub_agents=[greeting_agent, farewell_agent],
        output_key="last_weather_report",
        before_model_callback=block_keyword_guardrail, # Mantém o guardrail de modelo
        before_tool_callback=block_paris_tool_guardrail # Adiciona o guardrail de ferrameta
    )
    print(f"✅ Root Agent '{root_agent_tool_guardrail.name}' created with BOTH callbacks.")

    # Criando o Runner reaproveitando o mesmo serviço de sessão stateful
    if 'session_service_stateful' in globals(): # Verifica se o serviço de sessão com estado existe
        runner_root_tool_guardrail = Runner(
            agent=root_agent_tool_guardrail,
            app_name=APP_NAME,
            session_service=session_service_stateful # Usa o mesmo serviço de sessão, preservando o estado e histórico
        )
        print(f"✅ Runner created for tool guardrail agent '{runner_root_tool_guardrail.agent.name}', using stateful session service.")
    else:
        print("❌ Cannot create runner. 'session_service_stateful' from Step 4/5 is missing.")

else:
    print("❌ Cannot create root agent with tool guardrail. Prerequisites missing.")

✅ Sub-Agent 'greeting_agent' redefined.
✅ Sub-Agent 'farewell_agent' redefined.
✅ Root Agent 'weather_agent_v6_tool_guardrail' created with BOTH callbacks.
✅ Runner created for tool guardrail agent 'weather_agent_v6_tool_guardrail', using stateful session service.


Testando a barreira de proteção do argumento da ferramenta

In [ ]:
import asyncio

if 'runner_root_tool_guardrail' in globals() and runner_root_tool_guardrail: # Verifica se o runner do root agent com tool guardrail existe no escopo global e está válido
    async def run_tool_guardrail_test():
        print("\n--- Testing Tool Argument Guardrail ('Paris' blocked) ---")

        interaction_func = lambda query: call_agent_async(query, # Cria uma função lambda para simplificar chamadas repetidas ao call_agent_async
                                                         runner_root_tool_guardrail,
                                                         USER_ID_STATEFUL, # Usa um ID de usuário existente
                                                         SESSION_ID_STATEFUL # Usa um ID de sessão existente
                                                        )
        # 1. Cidade permitida deve passar tanto pelo guardrail do modelo quanto pelo guardrail da tool
        print("--- Turn 1: Requesting weather in New York (expect allowed) ---")
        await interaction_func("What's the weather in New York?") # Envia o pedido de clima para New York

        # 2. A mensagem deve passar pelo guardrail do modelo, mas a execução da tool deve ser bloqueada pelo guardrail da tool
        print("\n--- Turn 2: Requesting weather in Paris (expect blocked by tool guardrail) ---")
        await interaction_func("How about Paris?") # O callback de ferramenta deve impedir isso

        # 3. Um novo pedido de cidade permitida deve voltar a funcionar normalmente
        print("\n--- Turn 3: Requesting weather in London (expect allowed) ---")
        await interaction_func("Tell me the weather in London.") # Envia o pedido de clima para Londres

    print("Attempting execution using 'await' (default for notebooks)...")
    await run_tool_guardrail_test() # Executa a função do teste, rodando os 3 turnos


    # Inspecionando o estado da sessão final depois da conversa
    print("\n--- Inspecting Final Session State (After Tool Guardrail Test) ---")
    final_session = await session_service_stateful.get_session(app_name=APP_NAME, # Recupera a sessão via API do session service para ler o estado
                                                         user_id=USER_ID_STATEFUL,
                                                         session_id= SESSION_ID_STATEFUL)
    if final_session: # Verifica se a sessão foi encontrada
        print(f"Tool Guardrail Triggered Flag: {final_session.state.get('guardrail_tool_block_triggered', 'Not Set (or False)')}") # Imprime a flag que o tool guardrail seta quando bloqueia Paris
        print(f"Last Weather Report: {final_session.state.get('last_weather_report', 'Not Set')}") # Imprime o valor salvo no output_key, espera ser o clima de Londres
        print(f"Temperature Unit: {final_session.state.get('user_preference_temperature_unit', 'Not Set')}") # Imprime a unidade de temperatura atual, deve continuar a ser Fahrenheit
    else:
        print("\n❌ Error: Could not retrieve final session state.")

else:
    print("\n⚠️ Skipping tool guardrail test. Runner ('runner_root_tool_guardrail') is not available.")

Attempting execution using 'await' (default for notebooks)...

--- Testing Tool Argument Guardrail ('Paris' blocked) ---
--- Turn 1: Requesting weather in New York (expect allowed) ---

>>> User Query: What's the weather in New York?
--- Callback: block_keyword_guardrail running for agent: weather_agent_v6_tool_guardrail ---
--- Callback: Inspecting last user message: 'What's the weather in New York?...' ---
--- Callback: Keyword not found. Allowing LLM call for weather_agent_v6_tool_guardrail. ---


_ResourceExhaustedError: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 56.855193669s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier